# CLIN_0043 · Derive Param Bmi

**Clinical · admiral · L2 · public seed 1000**

This notebook is locked to the task selected on the RBioBench review website. Run **Setup**, inspect the task-aware data check, write your own R solution, and then run **Check output**. A reference solution is available at the end if you need it.

> Write an R script for an R package task: perform derive param bmi using the admiral package. You may use functions from the admiral package or an output-equivalent implementation.
>
> Input: dataset.tsv
> Output: result.csv
>
>
> Required columns for result.csv: USUBJID, PARAMCD, AVAL
> Read input files from inputs/ using relative paths. Write only the required output file(s) under outputs/. Create outputs/ if needed. Do not write alternative filenames.

**Expected artifacts:** `outputs/result.csv`

## Input data · seed 1000

These are the actual files loaded by **Setup**. Large tables are previewed below.

### `inputs/dataset.tsv`

```text
USUBJID	PARAMCD	AVAL
01	HEIGHT	170
01	WEIGHT	70
02	HEIGHT	180
02	WEIGHT	80
```

In [ ]:
# @title Setup
task_id <- "CLIN_0043"
session_root <- "/content/rbiobench"
archive <- file.path(session_root, "rbiobench-site.tar.gz")
pack_dir <- file.path(session_root, "practice_pack")
workspace <- file.path(session_root, "workspace", task_id)
dir.create(session_root, recursive = TRUE, showWarnings = FALSE)

# The practice pack lives in the GitHub repo under colab/practice_pack.
if (!file.exists(archive)) {
  download.file(
    "https://codeload.github.com/xihaopark/RBioBench-site/tar.gz/refs/heads/main",
    archive, mode = "wb", quiet = TRUE
  )
}

unlink(pack_dir, recursive = TRUE, force = TRUE)
extract_dir <- file.path(session_root, "repo")
unlink(extract_dir, recursive = TRUE, force = TRUE)
untar(archive, exdir = extract_dir)
stopifnot(file.rename(file.path(extract_dir, "RBioBench-site-main", "colab", "practice_pack"), pack_dir))
unlink(extract_dir, recursive = TRUE, force = TRUE)
unlink(workspace, recursive = TRUE, force = TRUE)
dir.create(workspace, recursive = TRUE, showWarnings = FALSE)
status <- system2("python3", c(
  shQuote(file.path(pack_dir, "scripts", "materialize_case.py")),
  "--release", shQuote(pack_dir), "--task", task_id, "--workspace", shQuote(workspace)
))
stopifnot(status == 0)
setwd(workspace)
cat("Ready:", task_id, "\nInputs:", paste(list.files("inputs"), collapse = ", "), "\n")

## Explore the inputs

A short, task-relevant check of the data before you start.

In [ ]:
advs <- read.delim("inputs/dataset.tsv", check.names = FALSE)
print(advs)
cat("Parameter coverage by subject:\n")
print(xtabs(~ USUBJID + PARAMCD, advs))

## Your solution

Edit this native-R cell and run it. It should read from `inputs/` and write the required artifact to `outputs/`.

In [ ]:
# Write your R solution below, then run this cell.
dir.create("outputs", showWarnings = FALSE)

# Inputs: inputs/dataset.tsv
# Required output: outputs/outputs/result.csv
# Your code starts here:


In [ ]:
# @title Check output
score <- system2("python3", c(
  shQuote(file.path(pack_dir, "evaluator", "score_submission.py")),
  "--release", shQuote(pack_dir), "--task", task_id,
  "--outputs", shQuote(file.path(workspace, "outputs"))
), stdout = TRUE, stderr = TRUE)
raw_report <- paste(score, collapse = "\n")
report <- tryCatch(jsonlite::fromJSON(raw_report, simplifyVector = FALSE), error = function(e) NULL)
escape_html <- function(x) {
  x <- gsub("&", "&amp;", as.character(x), fixed = TRUE)
  x <- gsub("<", "&lt;", x, fixed = TRUE)
  x <- gsub(">", "&gt;", x, fixed = TRUE)
  x
}

if (is.null(report)) {
  IRdisplay::display_html(paste0(
    '<div style="border:1px solid #b42318;background:#fff4f2;padding:18px;border-radius:8px">',
    '<div style="font-size:20px;font-weight:700;color:#b42318">CHECK ERROR</div>',
    '<pre style="white-space:pre-wrap">', escape_html(raw_report), '</pre></div>'
  ))
} else if (isTRUE(report$pass)) {
  IRdisplay::display_html(paste0(
    '<div style="border:1px solid #1f7a46;background:#eefbf3;padding:18px;border-radius:8px">',
    '<div style="font-size:22px;font-weight:700;color:#176b3a">✓ PASS</div>',
    '<div style="margin-top:6px;color:#245b3c">All required artifacts match the reference checks.</div></div>'
  ))
} else {
  failures <- unlist(lapply(names(report$comparisons), function(file) {
    item <- report$comparisons[[file]]
    if (isTRUE(item$match)) return(NULL)
    reason <- if (is.null(item$reason)) "Output did not match the reference." else item$reason
    tier <- if (is.null(item$tier)) "" else paste0(' <span style="color:#777">[', escape_html(item$tier), ']</span>')
    paste0('<li style="margin:8px 0"><strong>', escape_html(file), '</strong>: ',
           escape_html(reason), tier, '</li>')
  }))
  IRdisplay::display_html(paste0(
    '<div style="border:1px solid #b42318;background:#fff4f2;padding:18px;border-radius:8px">',
    '<div style="font-size:22px;font-weight:700;color:#b42318">✕ FAIL</div>',
    '<div style="margin-top:6px;color:#7a271a">The output needs attention:</div>',
    '<ul style="margin:8px 0 0;padding-left:22px">', paste(failures, collapse = ""), '</ul></div>'
  ))
}

## Reference solution.R

If you are stuck or want to compare approaches, the exact frozen reference solution is below. Run **Setup** again before using it if you want a clean workspace.

In [ ]:
dataset <- read.delim(file.path("inputs", "dataset.tsv"), check.names = FALSE, stringsAsFactors = FALSE)
wide <- reshape(dataset, idvar = "USUBJID", timevar = "PARAMCD", direction = "wide")
result_df <- data.frame(
  USUBJID = wide$USUBJID,
  PARAMCD = "BMI",
  AVAL = as.numeric(wide$AVAL.WEIGHT) / (as.numeric(wide$AVAL.HEIGHT) / 100)^2,
  stringsAsFactors = FALSE
)

dir.create("outputs", showWarnings = FALSE)
write.csv(result_df, file.path("outputs", "result.csv"), row.names = FALSE)